导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True


本节把“主波束”从一组静态图片，升级成一个可计算的方向依赖响应模型。对射电干涉来说，主波束并不是一个附属修饰项，而是决定视场、灵敏度、谱形偏差以及校准复杂度的核心因素之一。

从测量方程角度看，主波束属于方向依赖 Jones 项。若我们把天线对某个天空方向 $(l,m)$ 的电场响应记为 $E(l,m)$，则功率波束可以写成

$$
P(l,m) = |E(l,m)|^2.
$$

对两面响应相同的天线、单极化、单点源的情形，主波束会把该方向上的天空亮度直接乘上 $P(l,m)$。因此，主波束并不只是“图像边缘变暗”这么简单，它会直接进入可见度形成过程。


***


## 7.5 主波束：从口径照明到方向依赖增益

这一节围绕五个问题展开：

- 口径照明如何决定主瓣宽度和旁瓣水平；
- 为什么波束宽度会随频率缩放；
- `E-Jones` 与 `P-Jones` 如何进入测量方程；
- 地平式天线为什么会让主波束相对天空旋转；
- 为什么主波束改正虽然恢复了通量，却会放大噪声。


In [ ]:
c = 299792458.0


def circular_aperture_illumination(n=256, radius=0.42, taper="uniform", blockage=0.0):
    x = np.linspace(-1.0, 1.0, n)
    X, Y = np.meshgrid(x, x)
    R = np.sqrt(X**2 + Y**2)
    mask = (R <= radius) & (R >= blockage * radius)

    if taper == "uniform":
        illum = mask.astype(float)
    elif taper == "gaussian":
        illum = np.exp(-(R / (0.55 * radius)) ** 2) * mask
    elif taper == "cosine":
        illum = np.cos(0.5 * np.pi * R / radius) ** 2 * mask
    else:
        raise ValueError(f"unknown taper: {taper}")

    return x, illum


def beam_from_aperture(illumination):
    field = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(illumination)))
    power = np.abs(field) ** 2
    field /= np.max(np.abs(field))
    power /= np.max(power)
    return field, power


def peak_sidelobe_db(power_cut, threshold=1e-4):
    center = len(power_cut) // 2
    below = np.where(power_cut[center:] < threshold)[0]
    start = center + (below[0] if len(below) else 8)
    search_window = power_cut[start : min(len(power_cut), start + 80)]
    return 10.0 * np.log10(search_window.max() + 1e-12)


def dish_fwhm_deg(freq_hz, diameter_m, factor=1.02):
    return factor * (c / np.asarray(freq_hz)) / diameter_m * 180.0 / np.pi


def gaussian_power_beam(l_deg, m_deg, fwhm_l_deg, fwhm_m_deg=None, pa_deg=0.0):
    fwhm_m_deg = fwhm_l_deg if fwhm_m_deg is None else fwhm_m_deg
    pa = np.deg2rad(pa_deg)
    l = np.asarray(l_deg)
    m = np.asarray(m_deg)
    l_rot = l * np.cos(pa) + m * np.sin(pa)
    m_rot = -l * np.sin(pa) + m * np.cos(pa)
    exponent = (l_rot / fwhm_l_deg) ** 2 + (m_rot / fwhm_m_deg) ** 2
    return np.exp(-4.0 * np.log(2.0) * exponent)


def gaussian_field_beam(l_deg, m_deg, fwhm_l_deg, fwhm_m_deg=None, pa_deg=0.0):
    return np.sqrt(gaussian_power_beam(l_deg, m_deg, fwhm_l_deg, fwhm_m_deg, pa_deg=pa_deg))


def parallactic_angle_rad(hour_angle_rad, latitude_rad, declination_rad):
    y = np.sin(hour_angle_rad)
    x = np.tan(latitude_rad) * np.cos(declination_rad) - np.sin(declination_rad) * np.cos(hour_angle_rad)
    return np.arctan2(y, x)


### 7.5.1 口径照明与主波束形状

在远场近似下，天线主波束本质上是口径照明函数的傅里叶变换。这个事实极其重要，因为它告诉我们：

- **主瓣宽度**由口径的“有效电尺寸”决定；
- **旁瓣水平**由口径边缘如何照明决定；
- 更强的 taper 往往能压低旁瓣，但代价是主瓣变宽、分辨率略降。

下面比较“均匀照明”和“高斯 taper 照明”的二维口径与波束。


In [ ]:
x, illum_uniform = circular_aperture_illumination(taper="uniform")
_, illum_tapered = circular_aperture_illumination(taper="gaussian")

_, power_uniform = beam_from_aperture(illum_uniform)
_, power_tapered = beam_from_aperture(illum_tapered)

center = power_uniform.shape[0] // 2
beam_coord = np.fft.fftshift(np.fft.fftfreq(power_uniform.shape[0], d=x[1] - x[0]))
cut_uniform = power_uniform[center]
cut_tapered = power_tapered[center]

uniform_sidelobe_db = peak_sidelobe_db(cut_uniform)
tapered_sidelobe_db = peak_sidelobe_db(cut_tapered)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].imshow(illum_uniform, origin="lower", extent=[x[0], x[-1], x[0], x[-1]], cmap="viridis")
axes[0, 0].set_title("Uniform aperture illumination")
axes[0, 0].set_xlabel("x / D")
axes[0, 0].set_ylabel("y / D")

axes[0, 1].imshow(illum_tapered, origin="lower", extent=[x[0], x[-1], x[0], x[-1]], cmap="viridis")
axes[0, 1].set_title("Tapered aperture illumination")
axes[0, 1].set_xlabel("x / D")
axes[0, 1].set_ylabel("y / D")

axes[1, 0].imshow(10.0 * np.log10(power_uniform + 1e-6), origin="lower", cmap="magma", vmin=-60.0, vmax=0.0)
axes[1, 0].set_title("Beam power (uniform taper)")
axes[1, 0].set_xlabel("image x")
axes[1, 0].set_ylabel("image y")

axes[1, 1].imshow(10.0 * np.log10(power_tapered + 1e-6), origin="lower", cmap="magma", vmin=-60.0, vmax=0.0)
axes[1, 1].set_title("Beam power (Gaussian taper)")
axes[1, 1].set_xlabel("image x")
axes[1, 1].set_ylabel("image y")

fig.tight_layout()
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(beam_coord, 10.0 * np.log10(cut_uniform + 1e-8), label="uniform")
ax.plot(beam_coord, 10.0 * np.log10(cut_tapered + 1e-8), label="gaussian taper")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-70.0, 2.0)
ax.set_xlabel("normalized image coordinate")
ax.set_ylabel("beam cut [dB]")
ax.set_title("Central beam cut")
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)

print(f"uniform taper peak sidelobe  ≈ {uniform_sidelobe_db:.2f} dB")
print(f"gaussian taper peak sidelobe ≈ {tapered_sidelobe_db:.2f} dB")


从结果可以读出一个典型权衡：

- 均匀照明给出更窄的主瓣，但旁瓣更高；
- taper 照明把口径边缘压低，旁瓣明显下降；
- 这种“主瓣变宽、旁瓣变低”的交换，在实际反射面与馈源设计中无处不在。

因此，主波束并不是一个随便画出来的二维高斯，它背后对应着具体的口径物理和硬件设计选择。


### 7.5.2 波束宽度随频率变化，且会引入谱偏差

对口径直径为 $D$ 的单天线，主瓣宽度通常近似满足

$$
\theta_{\rm FWHM} \propto \frac{\lambda}{D}.
$$

这意味着频率越高，主波束越窄。对宽带观测而言，这种频率依赖性会使离轴源在不同频率上被不同程度地衰减，从而把**本来属于仪器的色散效应**误写进天体源谱。


In [ ]:
freqs = np.linspace(1.0e9, 2.0e9, 256)
fwhm_deg = dish_fwhm_deg(freqs, diameter_m=25.0)

off_axis_deg = 0.25
power_gain = gaussian_power_beam(off_axis_deg, 0.0, fwhm_deg)

true_spectrum = (freqs / 1.0e9) ** (-0.8)
observed_spectrum = true_spectrum * power_gain

true_alpha = np.polyfit(np.log(freqs / 1.0e9), np.log(true_spectrum), 1)[0]
observed_alpha = np.polyfit(np.log(freqs / 1.0e9), np.log(observed_spectrum + 1e-30), 1)[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(freqs / 1e9, fwhm_deg, color="tab:blue", lw=2.0)
axes[0].set_xlabel("frequency [GHz]")
axes[0].set_ylabel("FWHM [deg]")
axes[0].set_title("Beam width shrinks with frequency")

axes[1].plot(freqs / 1e9, true_spectrum / true_spectrum[0], label="true source spectrum")
axes[1].plot(freqs / 1e9, observed_spectrum / observed_spectrum[0], label="seen through the beam")
axes[1].set_xlabel("frequency [GHz]")
axes[1].set_ylabel("normalized flux density")
axes[1].set_title("An off-axis source acquires beam chromaticity")
axes[1].legend()

fig.tight_layout()
plt.show()
plt.close(fig)

print("25 m dish beam width:")
for idx in [0, 128, 255]:
    print(
        f"  nu = {freqs[idx] / 1e9:.2f} GHz -> FWHM = {fwhm_deg[idx]:.3f} deg, "
        f"gain at 0.25 deg = {power_gain[idx]:.3f}"
    )

print(f"true spectral index     = {true_alpha:.2f}")
print(f"apparent spectral index = {observed_alpha:.2f}")


这个例子展示了宽带成像里非常关键的一点：如果你不做主波束建模与改正，离轴源的谱指数会被严重扭曲。于是“天空谱结构”和“仪器方向依赖响应”就会彼此混淆。

这也是宽带主波束校正、A-projection 和方向依赖标定在现代阵列中不可或缺的原因之一。


### 7.5.3 `E-Jones` 与 `P-Jones` 如何进入测量方程

在上一节 [7.2 RIME](7_2_rime.ipynb) 中，我们已经见过方向依赖 Jones 项的一般写法。若把某个方向上的场响应写成 $E_p(l,m)$，则对单源、单极化、两面相同天线而言，

$$
V_{pq}^{\rm obs}(u,v) = E_p(l,m) E_q^*(l,m) I(l,m) e^{-2\pi i (ul + vm)}.
$$

如果两面天线完全相同且波束响应为实数，那么 $E_pE_q^*$ 就退化为功率波束 $P(l,m)$。下面用一个简单的三源天空模型来看看主波束如何把“真实天空”改写成“表观天空”。


In [ ]:
source_lm = [
    (0.00, 0.00, 1.0),
    (0.35, 0.20, 0.6),
    (-0.45, -0.30, 0.4),
]

axis_deg = np.linspace(-0.8, 0.8, 201)
L, M = np.meshgrid(axis_deg, axis_deg)
fwhm_15ghz = dish_fwhm_deg(1.5e9, 25.0)
beam_map = gaussian_power_beam(L, M, fwhm_15ghz)

true_sky = np.zeros_like(beam_map)
apparent_sky = np.zeros_like(beam_map)
source_gains = []

for l_deg, m_deg, flux in source_lm:
    j = np.argmin(np.abs(axis_deg - l_deg))
    i = np.argmin(np.abs(axis_deg - m_deg))
    true_sky[i, j] = flux

    E = gaussian_field_beam(l_deg, m_deg, fwhm_15ghz)
    apparent_sky[i, j] = flux * (E * np.conj(E)).real
    source_gains.append((l_deg, m_deg, flux, float((E * np.conj(E)).real)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

im0 = axes[0].imshow(true_sky, origin="lower", extent=[axis_deg[0], axis_deg[-1], axis_deg[0], axis_deg[-1]], cmap="viridis")
axes[0].set_title("True sky brightness")
axes[0].set_xlabel("l [deg]")
axes[0].set_ylabel("m [deg]")

im1 = axes[1].imshow(beam_map, origin="lower", extent=[axis_deg[0], axis_deg[-1], axis_deg[0], axis_deg[-1]], cmap="magma", vmin=0.0, vmax=1.0)
axes[1].set_title("Primary beam power")
axes[1].set_xlabel("l [deg]")
axes[1].set_ylabel("m [deg]")

im2 = axes[2].imshow(apparent_sky, origin="lower", extent=[axis_deg[0], axis_deg[-1], axis_deg[0], axis_deg[-1]], cmap="viridis")
axes[2].set_title("Apparent sky = P(l,m) I(l,m)")
axes[2].set_xlabel("l [deg]")
axes[2].set_ylabel("m [deg]")

fig.tight_layout()
plt.show()
plt.close(fig)

print(f"Zero-baseline true flux     = {true_sky.sum():.3f}")
print(f"Zero-baseline apparent flux = {apparent_sky.sum():.3f}")
print("\nBeam attenuation for each source:")
for l_deg, m_deg, flux, gain in source_gains:
    print(f"  source at ({l_deg:+.2f}, {m_deg:+.2f}) deg, flux={flux:.2f} -> P = {gain:.3f}")


这里的“表观天空”并不是后期图像处理里凭空制造出来的，而是相关器一开始就看到的天空。换句话说：

- 先被主波束加权的是天空；
- 后做傅里叶变换得到的才是观测可见度；
- 因此主波束误差会直接写进可见度域，而不是只在成图时才出现。

这也是为什么宽场成像不能简单依赖一个统一的标量增益，而必须认真对待方向依赖校准项。


### 7.5.4 地平式安装、视差角与波束旋转

大多数现代大阵列采用地平式（alt-az）安装。这意味着天线本体相对地面保持固定，而天空坐标系会随着跟踪不断相对旋转。对一个非圆对称波束而言，这种旋转会让固定天空方向上的增益随时间变化。

视差角（parallactic angle）是描述这种相对旋转的关键量，其常见表达式为

$$
\tan \chi = \frac{\sin H}{\tan \phi \cos \delta - \sin \delta \cos H},
$$

其中 $H$ 是时角，$\phi$ 是台址纬度，$\delta$ 是源赤纬。


In [ ]:
latitude_deg = -30.0
declination_deg = -40.0
hour_angle_hours = np.linspace(-3.0, 3.0, 181)

chi_deg = np.rad2deg(
    np.unwrap(
        parallactic_angle_rad(
            np.deg2rad(15.0 * hour_angle_hours),
            np.deg2rad(latitude_deg),
            np.deg2rad(declination_deg),
        )
    )
)

beam_major_deg = 0.75
beam_minor_deg = 0.55
sky_positions = [(0.30, 0.15), (-0.25, 0.28)]

gain_curves = []
for l_deg, m_deg in sky_positions:
    gain = gaussian_power_beam(l_deg, m_deg, beam_major_deg, beam_minor_deg, pa_deg=chi_deg)
    gain_curves.append((l_deg, m_deg, gain))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(hour_angle_hours, chi_deg, color="tab:blue", lw=2.0)
axes[0].set_xlabel("hour angle [h]")
axes[0].set_ylabel("parallactic angle [deg]")
axes[0].set_title("Parallactic angle during tracking")

for l_deg, m_deg, gain in gain_curves:
    axes[1].plot(hour_angle_hours, gain, lw=2.0, label=f"source ({l_deg:+.2f}, {m_deg:+.2f}) deg")
axes[1].set_xlabel("hour angle [h]")
axes[1].set_ylabel("primary-beam gain")
axes[1].set_title("A rotating elliptical beam modulates sky sources")
axes[1].legend()

fig.tight_layout()
plt.show()
plt.close(fig)

print(f"parallactic angle range = [{chi_deg.min():.1f}, {chi_deg.max():.1f}] deg")
for l_deg, m_deg, gain in gain_curves:
    print(
        f"source ({l_deg:+.2f}, {m_deg:+.2f}) deg -> "
        f"gain range [{gain.min():.3f}, {gain.max():.3f}]"
    )


这正是“地平式天线的波束相对天空旋转”在数值上的直接体现。若波束完全圆对称，那么旋转不会改变增益；但真实系统中常常存在轻微椭圆、馈源偏置、支撑腿遮挡、偏振差异等非圆对称结构，因此视差角旋转会变成真实的时间依赖误差。

这一步同时也是偏振成像中许多方向依赖效应的入口，不过更完整的偏振讨论会留到后面的偏振章节。


### 7.5.5 主波束改正会恢复通量，也会放大噪声

若图像平面上的观测结果为

$$
I_{\rm obs}(l,m) = P(l,m) I_{\rm true}(l,m) + n(l,m),
$$

那么最朴素的主波束改正就是除以 $P(l,m)$：

$$
I_{\rm corr}(l,m) = \frac{I_{\rm obs}(l,m)}{P(l,m)}.
$$

这能把离轴源的通量拉回去，但同时也会把噪声放大成 $\sigma_{\rm corr} \approx \sigma / P$。因此成像软件通常不会盲目改正到无限远，而会设置一个主波束截断半径。


In [ ]:
extent_deg = 1.2
n = 201
axis_deg = np.linspace(-extent_deg, extent_deg, n)
L, M = np.meshgrid(axis_deg, axis_deg)

beam = gaussian_power_beam(L, M, 0.70)
true_image = np.zeros_like(beam)

for l_deg, m_deg, flux in source_lm:
    j = np.argmin(np.abs(axis_deg - l_deg))
    i = np.argmin(np.abs(axis_deg - m_deg))
    true_image[i, j] = flux

rng = np.random.default_rng(9)
noise = 0.02 * rng.normal(size=true_image.shape)
observed_image = true_image * beam + noise

corrected_image = np.full_like(observed_image, np.nan)
valid = beam > 0.12
corrected_image[valid] = observed_image[valid] / beam[valid]

center_rms = np.nanstd(corrected_image[(beam > 0.75) & (true_image == 0.0)])
edge_rms = np.nanstd(corrected_image[(beam > 0.15) & (beam < 0.30) & (true_image == 0.0)])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

axes[0].imshow(true_image, origin="lower", extent=[axis_deg[0], axis_deg[-1], axis_deg[0], axis_deg[-1]], cmap="viridis")
axes[0].set_title("True sky")
axes[0].set_xlabel("l [deg]")
axes[0].set_ylabel("m [deg]")

axes[1].imshow(observed_image, origin="lower", extent=[axis_deg[0], axis_deg[-1], axis_deg[0], axis_deg[-1]], cmap="viridis")
axes[1].set_title("Observed image = P I + noise")
axes[1].set_xlabel("l [deg]")
axes[1].set_ylabel("m [deg]")

axes[2].imshow(np.nan_to_num(corrected_image, nan=0.0), origin="lower", extent=[axis_deg[0], axis_deg[-1], axis_deg[0], axis_deg[-1]], cmap="viridis")
axes[2].set_title("Primary-beam corrected image")
axes[2].set_xlabel("l [deg]")
axes[2].set_ylabel("m [deg]")

fig.tight_layout()
plt.show()
plt.close(fig)

print(f"noise RMS near beam center = {center_rms:.4f}")
print(f"noise RMS near beam edge   = {edge_rms:.4f}")

print("\nRecovered source fluxes after correction:")
for l_deg, m_deg, flux in source_lm:
    j = np.argmin(np.abs(axis_deg - l_deg))
    i = np.argmin(np.abs(axis_deg - m_deg))
    print(
        f"  source ({l_deg:+.2f}, {m_deg:+.2f}) deg: "
        f"true={flux:.3f}, observed={observed_image[i, j]:.3f}, corrected={corrected_image[i, j]:.3f}"
    )


到这里，主波束这一节应该形成一个完整链条了：

- 口径照明决定主瓣和旁瓣；
- 波束宽度随频率缩放，使离轴源获得仪器谱结构；
- `E-Jones` 与 `P-Jones` 直接把天空改写为表观天空；
- 地平式安装通过视差角让非圆对称波束随时间旋转；
- 主波束改正虽然能恢复通量，却不可避免地放大边缘噪声。

对中国学生来说，真正需要建立的不是“主波束会让边缘变暗”这种口头印象，而是一个更严格的判断：**主波束是一个方向依赖、频率依赖、时间依赖，并且直接进入测量方程的系统响应项。** 一旦进入宽带、宽场和高动态范围成像，这个响应就必须被建模、校准并正确地传播到误差分析中。
